# Item 3, the 7B scale check, on Kaggle "GPU T4 x2" (D157; exploratory, not the paper)

The plan and its rules are in `docs/exploratory/DECISIONS_FOLLOWUP.md` D157 (repo commit `865c6fac8b1f`), and the
steps in `docs/exploratory/kaggle_instructions.md`.

The notebook:
- clones the repo at `COMMIT` and installs vLLM in its own venv;
- downloads `Qwen/Qwen2.5-Math-7B-Instruct` at its pinned revision and checks the sha256 of the weights;
- times the first 20 candidates and sizes N to fit inside 9.5 h;
- writes the samples in chunks of 24 candidates to `/kaggle/working`, so a cutoff loses at most one chunk;
- zips them.

**Settings:** Accelerator **GPU T4 x2**, Internet **On**. **Run with** Save Version → **Save & Run All (Commit)**.
**Do not edit the cells.** A resume needs only the previous version's output attached as an input.

In [ ]:
# Settings and the machine. Nothing to edit: COMMIT pins the code (D157).
import hashlib, json, os, shutil, signal, subprocess, sys, threading, time
from pathlib import Path

SESSION_START = time.time()                      # D157: sampling ends by SESSION_START + 9.5 h
COMMIT = "865c6fac8b1f8b2e77fb36f1bae3ebc63c7d03a8"
REPO_URL = "https://github.com/RaghhavMalani/operation-physics.git"
WORK = Path("/kaggle/working")                   # saved as the version's output
TMP = Path("/tmp/physcal")                       # repo, venv, weights: not saved
REPO, VENV, MODEL_DIR = TMP / "repo", TMP / "venv", TMP / "qwen2.5-math-7b-instruct"
OUT = WORK / "scale" / "7b"
(OUT / "logs").mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
LOG = OUT / "logs" / f"session_{int(SESSION_START)}.log"


def sh(cmd, tail=4000, **kw):
    """Run a shell command; its output goes to the cell (the last `tail` characters) and to the session log.
    A failure raises, which stops Save & Run All before anything is sampled."""
    print(f"$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True, **kw)
    out = r.stdout + r.stderr
    print(out[-tail:], flush=True)
    with LOG.open("a") as fh:
        fh.write(f"$ {cmd}\n{out}\n")
    if r.returncode:
        raise RuntimeError(f"exit {r.returncode}: {cmd}")
    return r.stdout


GPUS = [line.split(", ") for line in
        sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader").strip().splitlines()]
assert len(GPUS) == 2 and all("T4" in g[0] for g in GPUS), f"needs Accelerator 'GPU T4 x2', got {GPUS}"
sh("df -h /tmp /kaggle/working")

In [ ]:
# The repo at COMMIT (sparse: the code, the configs and X2's candidate pool), and the checks that need its config.
if not (REPO / ".git").exists():
    sh(f"git init -q {REPO}")
    sh(f"git -C {REPO} remote add origin {REPO_URL}")
    sh(f"git -C {REPO} sparse-checkout set --no-cone /physcal/ /configs/ "
       "/data/eval_sets/1.1.0+c735a0d4/x2_candidates.jsonl")
    sh(f"git -C {REPO} fetch -q --depth 1 --filter=blob:none origin {COMMIT}")
    sh(f"git -C {REPO} checkout -q FETCH_HEAD")
HEAD = sh(f"git -C {REPO} rev-parse HEAD").strip()
assert HEAD == COMMIT, (HEAD, COMMIT)
try:
    import yaml
except ImportError:
    sh(f"{sys.executable} -m pip install -q pyyaml==6.0.3")
    import yaml
KC = yaml.safe_load((REPO / "configs" / "followup.yaml").read_text())["scale"]["kaggle"]
DRIVER = min(int(g[1].split(".")[0]) for g in GPUS)
assert DRIVER >= KC["min_driver"], f"driver {DRIVER} < {KC['min_driver']}: the pinned vLLM needs CUDA 13"
FREE_GB = shutil.disk_usage("/tmp").free / 1e9
assert FREE_GB >= KC["min_free_gb"], f"less than {KC['min_free_gb']} GB free on /tmp ({FREE_GB:.0f} GB)"
CAND = REPO / "data/eval_sets/1.1.0+c735a0d4/x2_candidates.jsonl"
assert hashlib.sha256(CAND.read_bytes()).hexdigest() == KC["candidates_sha256"], "the candidate pool differs"
sys.path.insert(0, str(REPO))
from physcal.eval.followup_kaggle import EXIT_HEALTH  # noqa: E402
print(f"repo {HEAD} | candidates ok | driver {DRIVER} | {FREE_GB:.0f} GB free on /tmp", flush=True)

In [ ]:
# The venv (uv; isolated from Kaggle's preinstalled packages): vLLM and the parser's packages, pinned in the config.
os.environ["UV_CACHE_DIR"] = str(TMP / "uv-cache")
os.environ["UV_PYTHON_INSTALL_DIR"] = str(TMP / "uv-python")
sh(f"{sys.executable} -m pip install -q uv=={KC['uv']}")
UV = f"{sys.executable} -m uv"
if not (VENV / "bin" / "python").exists():
    sh(f"{UV} venv -q --python {KC['python']} {VENV}")
sh(f"{UV} pip install -q --python {VENV}/bin/python " + " ".join(KC["packages"]))
PY = str(VENV / "bin" / "python")
ENV = {**os.environ, "PYTHONPATH": str(REPO), "HF_HOME": str(TMP / "hf"), "TOKENIZERS_PARALLELISM": "false",
       "VLLM_WORKER_MULTIPROC_METHOD": "spawn", "VLLM_USE_FLASHINFER_SAMPLER": "0"}  # no FlashInfer kernels on sm75
sh(f"{PY} -c \"import torch, vllm; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'gpus', "
   f"torch.cuda.device_count(), [torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())], "
   f"'vllm', vllm.__version__)\"", env=ENV)

In [ ]:
# A previous version's output attached as an input (a resume), then the weights at the pinned revision (sha256-checked).
def fk(args):
    return sh(f"{PY} -m physcal.eval.followup_kaggle {args}", env=ENV, cwd=str(TMP))


fk(f"restore --out {WORK} --src /kaggle/input")
if not (MODEL_DIR / "_verified.json").exists():
    fk(f"download --model-dir {MODEL_DIR}")
print((MODEL_DIR / "_verified.json").read_text())

In [ ]:
# Sampling: the sampler is its own process group; its lines stream here and into the session log. It is killed after
# stall_minutes of silence, and any failure is retried once with enforce_eager and NCCL_P2P_DISABLE=1 (D157). This
# cell never raises, so the zip below is always written.
def run_sampler(extra, env):
    cmd = [PY, "-m", "physcal.eval.followup_kaggle", "sample", "--model-dir", str(MODEL_DIR), "--out", str(WORK),
           "--session-start", repr(SESSION_START)] + extra
    print("$ " + " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd=str(TMP),
                         start_new_session=True, bufsize=1)
    last = [time.time()]

    def pump():
        with LOG.open("a") as fh:
            for line in p.stdout:
                last[0] = time.time()
                print(line, end="", flush=True)
                fh.write(line)
                fh.flush()

    t = threading.Thread(target=pump, daemon=True)
    t.start()
    while p.poll() is None:
        time.sleep(30)
        if time.time() - last[0] > KC["stall_minutes"] * 60:
            print(f"STALL: no output for {KC['stall_minutes']} min; killing the sampler", flush=True)
            os.killpg(p.pid, signal.SIGKILL)
            p.wait()
    t.join(timeout=60)
    return p.returncode


ATTEMPTS = [([], {}), (["--enforce-eager"], {"NCCL_P2P_DISABLE": "1"})][:KC["attempts"]]
STATUS = None
for i, (extra, env_extra) in enumerate(ATTEMPTS):
    try:
        STATUS = run_sampler(extra, {**ENV, **env_extra})
    except Exception as exc:
        print(f"attempt {i + 1} could not run: {exc!r}", flush=True)
        STATUS = -1
    print(f"attempt {i + 1}: exit {STATUS}", flush=True)
    if STATUS in (0, EXIT_HEALTH) or time.time() > SESSION_START + KC["sampling_hours"] * 3600:
        break

In [ ]:
# The zip (chunks, meta, timing, plan, logs, and a MANIFEST of sha256s) for the local ingest, and the status line.
fk(f"pack --out {WORK}")
for name in ("throughput.json", "plan.json"):
    f = OUT / name
    print(name, f.read_text() if f.exists() else "missing", flush=True)
N_CHUNKS = len(list((OUT / "samples").glob("chunk_*.jsonl")))
print(f"STATUS: sampler exit {STATUS} (0 done, {EXIT_HEALTH} health gate); {N_CHUNKS} chunks = {N_CHUNKS * 24} "
      f"candidates; download {WORK / KC['zip_name']} from the Output tab", flush=True)